# AMLH 2025-26 Coursework — Patient Question Classification (Dataset C)

Predict the disease associated with a patient question, from **906** candidate labels.
Train: 8,891 questions / 906 classes. Test: 200 questions / 102 classes (all seen in training).

### What the audit established about this dataset

| Property | Finding | Design consequence |
|---|---|---|
| Class support | Near-**uniform**: min 4, median 10, max 20 (sd 0.98) | Not a long-tail problem. It is **extreme multi-class + few-shot** |
| Label granularity | 355/906 labels sit in a shared-prefix family (`baby_` 53, `pregnancy_` 24, `social_` 23, `cosmetic_` 20, `contraception_` 14) | Fine-grained, partly overlapping labels cap single-label accuracy |
| Label ambiguity | 85 identical question strings map to >1 disease | An irreducible error floor exists |
| Train-test duplication | Only 0.5% of test questions exceed 0.9 cosine to any train question | Benchmark is honest; report as a validity check |
| Question length | mean 8.4 words, 99th pct 18, max 29 | `max_length=48` is ample for BERT |
| **Sibling homogeneity** | train→sibling cosine **0.572**, test→own-class **0.356** | **Held-out validation is optimistic by construction** (see §0.5) |

### Systems compared

| Arm | Approach | Lab lineage |
|---|---|---|
| 1 | TF-IDF retrieval + similarity-weighted k-NN vote | Week 7 (NLP1), Week 10 (NLP4) |
| 2 | Fine-tuned BERT classifier, 906-way head | Week 9 (NLP3) |
| 3 | LLM selection over a retrieval-shortlisted candidate set | Week 10 (NLP4) |

**Headline metric is accuracy**, as the brief specifies. Top-5 and MRR appear only in the error
analysis, where they explain *why* accuracy is capped.

**Runtime.** Colab T4. Arm 1 is CPU-only. Set `USE_API = True` in Arm 3 if no GPU is available.

## 0. Setup

In [ ]:
# Colab only.
# from google.colab import drive; drive.mount('/content/drive')
# %cd /content/drive/MyDrive/amlh_nlp_coursework

!pip -q install transformers datasets scikit-learn seaborn spacy 2>/dev/null

In [ ]:
# ============================ CONFIG & REPRODUCIBILITY ============================
import os, re, json, random, itertools, sys, warnings
from collections import defaultdict
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
warnings.filterwarnings("ignore")

SEED = 42

def set_seed(seed=SEED):
    random.seed(seed); np.random.seed(seed); os.environ["PYTHONHASHSEED"] = str(seed)
    try:
        import torch
        torch.manual_seed(seed)
        if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)
    except ImportError:
        pass

set_seed()

DATA_DIR   = "."
SEARCH_DIR = "./search_data"          # unzipped db_nhs_qa_classification.zip
ART_DIR    = "./artefacts"; os.makedirs(ART_DIR, exist_ok=True)
FIG_DIR    = "./figures";   os.makedirs(FIG_DIR, exist_ok=True)

sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.width", 170)

import sklearn
env = {"python": sys.version.split()[0], "numpy": np.__version__,
       "pandas": pd.__version__, "sklearn": sklearn.__version__}
try:
    import torch, transformers
    env |= {"torch": torch.__version__, "transformers": transformers.__version__,
            "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"}
except ImportError:
    env["gpu"] = "not checked"
json.dump(env, open(f"{ART_DIR}/environment.json", "w"), indent=2); env

---
## Phase 0 — Loading, integrity audit, and validation design

Three checks precede any modelling, because each can invalidate the comparison:
schema and label-space integrity; **feature isolation** (`answer` is a reference answer, not an
input — using it at inference is leakage, though training-side answers are legitimate *class*
evidence); and train-test near-duplication, since the corpus is synthetically generated.

In [ ]:
# ------------------------------ 0.1 LOAD + SCHEMA CHECK ------------------------------
train_df = pd.read_csv(f"{DATA_DIR}/patient_qa_classification_train.csv")
test_df  = pd.read_csv(f"{DATA_DIR}/patient_qa_classification_test.csv")

EXPECTED = {"question", "disease", "answer", "reference_url"}
for name, d in [("train", train_df), ("test", test_df)]:
    assert not (EXPECTED - set(d.columns)), f"{name} missing {EXPECTED - set(d.columns)}"
    print(f"{name:5s}: {len(d):5d} rows | {d.disease.nunique():4d} classes | "
          f"nulls {d.isna().sum().sum()}")

# CONTRACT: `question` is the only inference-time input, for every arm.
INFERENCE_FEATURES = ["question"]
train_df = train_df.dropna(subset=["question", "disease"]).reset_index(drop=True)
test_df  = test_df.dropna(subset=["question", "disease"]).reset_index(drop=True)
train_df.head(3)

In [ ]:
# ------------------------------ 0.2 INTEGRITY AUDIT ------------------------------
cnt = train_df.disease.value_counts()
audit = {
    "n_train": len(train_df), "n_test": len(test_df),
    "n_classes_train": int(train_df.disease.nunique()),
    "n_classes_test":  int(test_df.disease.nunique()),
    "test_labels_unseen_in_train": sorted(set(test_df.disease) - set(train_df.disease)),
    "class_support": {"min": int(cnt.min()), "median": float(cnt.median()),
                      "mean": round(float(cnt.mean()), 2), "max": int(cnt.max()),
                      "sd": round(float(cnt.std()), 2),
                      "n_singletons": int((cnt == 1).sum()),
                      "n_below_5": int((cnt < 5).sum())},
    "exact_dup_question_disease": int(train_df.duplicated(["question", "disease"]).sum()),
    "duplicate_question_strings": int(train_df.duplicated(["question"]).sum()),
}

# Ambiguity: the same question string mapped to more than one disease.
g = train_df[train_df.duplicated("question", keep=False)].groupby("question").disease.nunique()
audit["ambiguous_question_strings"] = int((g > 1).sum())

# Label granularity: shared-prefix families.
fam = pd.Series(sorted(train_df.disease.unique())).str.split("_").str[0]
audit["labels_in_a_family"] = int((fam.map(fam.value_counts()) > 1).sum())
audit["largest_families"] = fam.value_counts().head(5).to_dict()

print(json.dumps(audit, indent=2))
print("\nExample ambiguous questions:")
for q in g[g > 1].index[:4]:
    print(f"  '{q}' -> {sorted(train_df.loc[train_df.question == q, 'disease'])[:3]}")

In [ ]:
# ------------------- 0.3 TRAIN-TEST NEAR-DUPLICATE AUDIT -------------------
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

dup_vec = TfidfVectorizer(ngram_range=(1, 2), sublinear_tf=True).fit(train_df.question)
sim = cosine_similarity(dup_vec.transform(test_df.question), dup_vec.transform(train_df.question))
max_sim, arg_sim = sim.max(axis=1), sim.argmax(axis=1)

audit["near_duplication"] = {f"frac_cos_ge_{t}": round(float((max_sim >= t).mean()), 4)
                             for t in (0.95, 0.90, 0.80, 0.70)}
audit["near_duplication"]["median_max_cosine"] = round(float(np.median(max_sim)), 4)
print(json.dumps(audit["near_duplication"], indent=2))

dup_report = pd.DataFrame({
    "test_question": test_df.question, "gold": test_df.disease,
    "nearest_train_question": train_df.question.values[arg_sim],
    "nearest_train_label":    train_df.disease.values[arg_sim],
    "cosine": max_sim.round(3)}).sort_values("cosine", ascending=False)
dup_report.to_csv(f"{ART_DIR}/near_duplicate_audit.csv", index=False)
dup_report.head(5)

### 0.4 Validation split — simple stratified hold-out

Class support has a minimum of 4, so **stratified splitting is feasible**. The validation set is
built to the same size (200) and class count (102) as the test set, so hyperparameters are chosen
under a comparable decision load. This follows the Week 9 (NLP3) protocol.

Its limitations are quantified in §0.5 and must be stated in the report rather than glossed over.

In [ ]:
# ------------------------------ 0.4 VALIDATION SPLIT ------------------------------
VAL_SIZE, VAL_N_CLASSES = len(test_df), test_df.disease.nunique()
rng = np.random.default_rng(SEED)

eligible    = cnt[cnt >= 3].index.to_numpy()
val_classes = rng.choice(eligible, size=VAL_N_CLASSES, replace=False)

val_idx = []
for c in val_classes:                     # mirrors the test set's ~2 questions per class
    pool = train_df.index[train_df.disease == c].to_numpy()
    take = min(2, len(pool) - 2)          # always leave >= 2 examples in the fit set
    if take > 0:
        val_idx += list(rng.choice(pool, size=take, replace=False))
val_idx = list(rng.permutation(val_idx))[:VAL_SIZE]

val_df = train_df.loc[val_idx].reset_index(drop=True)
fit_df = train_df.drop(index=val_idx).reset_index(drop=True)
assert set(val_df.disease) <= set(fit_df.disease)
print(f"fit={len(fit_df)} | val={len(val_df)} ({val_df.disease.nunique()} classes) | test={len(test_df)}")

fit_df.to_csv(f"{ART_DIR}/split_fit.csv", index=False)
val_df.to_csv(f"{ART_DIR}/split_val.csv", index=False)

### 0.5 Why the hold-out is optimistic — sibling phrasing homogeneity

The ~10 questions per disease were generated in a single pass, so they share vocabulary and
phrasing. A validation question carved from training is therefore a *sibling* of the questions
left behind, while a real test question is not. Measured on this dataset:

| Quantity | Cosine |
|---|---|
| Train question → nearest same-class sibling | ~0.57 |
| Validation question → nearest own-class training question | ~0.57 |
| **Test question → nearest own-class training question** | **~0.36** |
| Test question → nearest *wrong*-class training question | ~0.45 |

For roughly **62% of test questions a wrong-class training question is more similar** than any
right-class one. Consequence: hold-out validation over-estimates test accuracy by tens of points
(~0.77 vs ~0.40 for Arm 1). Note that k-fold cross-validation would **not** fix this — siblings
remain inside every training fold. The bias is a property of data generation, not of slicing.

Two surface explanations were tested and **rejected**: acronym-bearing test questions are *easier*
(0.69 vs 0.27), and post-stratifying validation to the test set's acronym × length marginals moves
the estimate the wrong way (0.765 → 0.795). Report this — a rejected hypothesis is real analysis.

**Secondary diagnostic (§0.6).** Pruning near-siblings from the index until validation novelty
matches the test level recovers a realistic estimate. Report it briefly alongside the hold-out
numbers, not as the primary protocol.

In [ ]:
# --------------- 0.5 SIBLING HOMOGENEITY MEASUREMENT (report this table) ---------------
hom_vec = TfidfVectorizer(sublinear_tf=True).fit(train_df.question)
Xtr, Xte = hom_vec.transform(train_df.question), hom_vec.transform(test_df.question)

sib = []
for d, idx in train_df.groupby("disease").groups.items():
    idx = np.array(idx)
    if len(idx) < 2: continue
    S = cosine_similarity(Xtr[idx]); np.fill_diagonal(S, -1)
    sib.extend(S.max(1).tolist())

by = train_df.groupby("disease").groups
own       = [float(cosine_similarity(Xte[i], Xtr[np.array(by[d])]).max())
             for i, d in enumerate(test_df.disease)]
S_all     = cosine_similarity(Xte, Xtr)
other     = [float(S_all[i][train_df.disease.values != d].max())
             for i, d in enumerate(test_df.disease)]

homogeneity = pd.DataFrame([
    {"comparison": "train question -> nearest same-class sibling", "mean_cosine": np.mean(sib)},
    {"comparison": "test question -> nearest own-class train q",   "mean_cosine": np.mean(own)},
    {"comparison": "test question -> nearest other-class train q", "mean_cosine": np.mean(other)},
]).round(3)
homogeneity["note"] = ["", "", f"wrong class closer for "
                       f"{np.mean(np.array(other) > np.array(own)):.1%} of test questions"]
homogeneity.to_csv(f"{ART_DIR}/sibling_homogeneity.csv", index=False)
homogeneity

In [ ]:
# --------------- 0.6 SECONDARY DIAGNOSTIC: novelty-calibrated validation ---------------
def novelty_calibrated_eval(prune_threshold, k=20, vec_kwargs=None):
    """Drop fit questions that are near-siblings of the validation question, so that
    validation novelty matches the test set's (~0.36 median own-class similarity)."""
    vec = TfidfVectorizer(**(vec_kwargs or dict(ngram_range=(1,1), sublinear_tf=True)))
    Xf  = vec.fit_transform(fit_df.question); Xq = vec.transform(val_df.question)
    S   = cosine_similarity(Xq, Xf)
    fl  = fit_df.disease.values
    ok, nov = [], []
    for i, gold in enumerate(val_df.disease):
        s = S[i].copy(); keep = s < prune_threshold; s[~keep] = -1
        votes = defaultdict(float)
        for j in np.argsort(-s)[:k]:
            if s[j] > 0: votes[fl[j]] += s[j]
        ok.append(max(votes, key=votes.get) == gold if votes else False)
        o = S[i][(fl == gold) & keep]
        nov.append(float(o.max()) if o.size else 0.0)
    return round(float(np.mean(ok)), 3), round(float(np.median(nov)), 3)

test_novelty = round(float(np.median(own)), 3)
rows = [{"prune_threshold": t, "val_accuracy": a, "median_own_class_novelty": n}
        for t in [1.01, 0.70, 0.60, 0.50, 0.45, 0.40]
        for a, n in [novelty_calibrated_eval(t)]]
calib = pd.DataFrame(rows)
print(f"target novelty (test) = {test_novelty}")
calib.to_csv(f"{ART_DIR}/novelty_calibration.csv", index=False)
calib

In [ ]:
# ------------------------------ SAVE AUDIT ------------------------------
audit["sibling_homogeneity"] = homogeneity.set_index("comparison").mean_cosine.to_dict()
audit["test_median_own_class_novelty"] = test_novelty
json.dump(audit, open(f"{ART_DIR}/integrity_audit.json", "w"), indent=2)
print("audit saved")

---
## Phase 1 — Exploratory data analysis

Figures are printed here but the marks sit in the *Preprocessing* band, so cross-reference them
from §2.1/§2.2 of the report as well as §3.1.

In [ ]:
# ------------------------------ 1.1 EDA FIGURES ------------------------------
for d in (train_df, test_df):
    d["q_len"] = d.question.str.split().str.len()

fig, ax = plt.subplots(2, 2, figsize=(13, 9))

ax[0, 0].plot(range(1, len(cnt) + 1), cnt.values)
ax[0, 0].set(xlabel="Disease rank", ylabel="Training questions",
             title=f"(a) Near-uniform class support ({len(cnt)} classes)")

sns.histplot(train_df.q_len, bins=30, stat="density", ax=ax[0, 1], label="train")
sns.histplot(test_df.q_len, bins=30, stat="density", ax=ax[0, 1], label="test", color="darkorange")
ax[0, 1].legend(); ax[0, 1].set(xlabel="Question length (words)", title="(b) Question length")

fam_counts = fam.value_counts()
fam_counts.head(12).plot.barh(ax=ax[1, 0])
ax[1, 0].invert_yaxis(); ax[1, 0].set(xlabel="Labels in family", title="(c) Label-family granularity")

ax[1, 1].hist(max_sim, bins=30); ax[1, 1].axvline(0.9, ls="--", c="r")
ax[1, 1].set(xlabel="Max cosine to any training question",
             title="(d) Train-test near-duplication (low)")

plt.tight_layout(); plt.savefig(f"{FIG_DIR}/fig1_eda.png", dpi=150); plt.show()
print(train_df.q_len.describe(percentiles=[.5, .9, .95, .99]))   # -> justifies BERT max_length

In [ ]:
# ------------- 1.2 NOVELTY DISTRIBUTIONS: the central figure of the report -------------
plt.figure(figsize=(8, 5))
sns.kdeplot(sib,   label="train q -> same-class sibling",     fill=True)
sns.kdeplot(own,   label="test q -> own-class train q",       fill=True)
sns.kdeplot(other, label="test q -> other-class train q",     fill=True)
plt.xlabel("Cosine similarity"); plt.title("Sibling phrasing homogeneity explains the val-test gap")
plt.legend(); plt.tight_layout(); plt.savefig(f"{FIG_DIR}/fig2_novelty.png", dpi=150); plt.show()

In [ ]:
# ---------------- 1.3 PCA OF TF-IDF QUESTION VECTORS (Week 7 style) ----------------
from sklearn.decomposition import PCA

big  = cnt.head(10).index
sub  = train_df[train_df.disease.isin(big)]
Xp   = TfidfVectorizer(sublinear_tf=True, min_df=2).fit_transform(sub.question)
pcs  = PCA(n_components=2, random_state=SEED).fit_transform(Xp.toarray())

plt.figure(figsize=(8, 6))
sns.scatterplot(x=pcs[:, 0], y=pcs[:, 1], hue=sub.disease.values, s=25)
plt.title("PCA of TF-IDF question vectors, 10 largest classes")
plt.legend(bbox_to_anchor=(1.02, 1), loc="upper left", fontsize=7)
plt.tight_layout(); plt.savefig(f"{FIG_DIR}/fig3_pca.png", dpi=150); plt.show()

---
## Phase 2 — Arm 1: TF-IDF retrieval + k-NN label vote

Retrieve the *k* nearest training texts by cosine similarity and vote for their labels
**weighted by similarity** — a plain majority vote discards the information that a 0.9-similarity
neighbour is stronger evidence than a 0.3 one.

Four index variants, each adding a source of class evidence:

| Variant | Index contents |
|---|---|
| Q | Training questions only |
| Q+L | + label name as text (`face_blindness` → "face blindness") |
| Q+L+A | + **training** reference answers (never test answers) |
| Q+L+A+D | + cleaned NHS class document from the ZIP |

**On the `answer` column.** Including training answers is not leakage: no test-side information
enters, and at inference the only input is the test *question*. It is exactly analogous to
indexing a knowledge base. State this explicitly in §2.3 — a marker will otherwise wonder.

In [ ]:
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics import f1_score

def knn_rank(index_texts, index_labels, query_texts, k=20, vec_kwargs=None):
    """Return (ranked labels, top similarity) per query."""
    vec = TfidfVectorizer(**(vec_kwargs or {}))
    Xf  = vec.fit_transform(index_texts); Xq = vec.transform(query_texts)
    nn  = NearestNeighbors(n_neighbors=min(k, Xf.shape[0]), metric="cosine").fit(Xf)
    dist, idx = nn.kneighbors(Xq)
    ranked, top = [], []
    for drow, irow in zip(dist, idx):
        votes = defaultdict(float)
        for dd, ii in zip(drow, irow):
            votes[index_labels[ii]] += (1.0 - dd)       # similarity-weighted vote
        ranked.append([l for l, _ in sorted(votes.items(), key=lambda x: -x[1])] or ["<abstain>"])
        top.append(float(1.0 - drow[0]))
    return ranked, top

def score_ranked(ranked, gold):
    """Accuracy is the headline metric. acc@5 / MRR are diagnostic only (see §3.3)."""
    top1 = [r[0] for r in ranked]
    return {"accuracy": round(float(np.mean([p == g for p, g in zip(top1, gold)])), 4),
            "acc@5":    round(float(np.mean([g in r[:5] for r, g in zip(ranked, gold)])), 4),
            "macro_f1": round(float(f1_score(gold, top1, average="macro", zero_division=0)), 4),
            "mrr":      round(float(np.mean([1/(r.index(g)+1) if g in r else 0.0
                                             for r, g in zip(ranked, gold)])), 4)}

In [ ]:
# ---------------- 2.1 HYPERPARAMETER GRID (validation) ----------------
GRID = {"ngram_range":  [(1, 1), (1, 2)],
        "sublinear_tf": [False, True],      # sklearn default False
        "min_df":       [1, 2],             # sklearn default 1
        "stop_words":   [None, "english"]}  # sklearn default None
K_VALUES = [1, 3, 5, 10, 20, 30]

rows = []
for combo in itertools.product(*GRID.values()):
    cfg = dict(zip(GRID, combo))
    for k in K_VALUES:
        r, _ = knn_rank(fit_df.question.tolist(), fit_df.disease.tolist(),
                        val_df.question.tolist(), k=k, vec_kwargs=cfg)
        rows.append({**{a: str(b) for a, b in cfg.items()}, "k": k,
                     **score_ranked(r, val_df.disease.tolist())})

grid_results = pd.DataFrame(rows).sort_values("accuracy", ascending=False).reset_index(drop=True)
grid_results.to_csv(f"{ART_DIR}/arm1_grid_validation.csv", index=False)

# CAVEAT for the report: 200 validation items => SE ~3.5pp. Differences under ~7pp are noise.
print(f"validation SE ~ {np.sqrt(0.77*0.23/len(val_df)):.3f}")
grid_results.head(10)

In [ ]:
# ---------------- 2.2 PREPROCESSING ABLATION (experiment, not assumption) ----------------
import spacy
try:
    nlp = spacy.load("en_core_web_sm", disable=["parser", "ner"])
except OSError:
    !python -m spacy download en_core_web_sm
    nlp = spacy.load("en_core_web_sm", disable=["parser", "ner"])

def lemmatise(texts, drop_stop=True):
    return [" ".join(t.lemma_.lower() for t in doc
                     if t.is_alpha and not (drop_stop and t.is_stop))
            for doc in nlp.pipe(list(texts), batch_size=256)]

best = grid_results.iloc[0]
VEC_BEST = dict(ngram_range=eval(best.ngram_range),
                sublinear_tf=best.sublinear_tf == "True",
                min_df=int(best.min_df),
                stop_words=None if best.stop_words == "None" else "english")
K_BEST = int(best.k)
print("selected:", VEC_BEST, "k =", K_BEST)

rows = []
for name, tf, vf in [
        ("raw", fit_df.question.tolist(), val_df.question.tolist()),
        ("lemma + stop-word removal", lemmatise(fit_df.question), lemmatise(val_df.question)),
        ("lemma only", lemmatise(fit_df.question, False), lemmatise(val_df.question, False))]:
    r, _ = knn_rank(tf, fit_df.disease.tolist(), vf, k=K_BEST, vec_kwargs=VEC_BEST)
    rows.append({"preprocessing": name, **score_ranked(r, val_df.disease.tolist())})

preproc_ablation = pd.DataFrame(rows)
preproc_ablation.to_csv(f"{ART_DIR}/arm1_preprocessing_ablation.csv", index=False)
preproc_ablation

In [ ]:
# ---------------- 2.3 INDEX VARIANTS: label text, answers, NHS documents ----------------
BOILERPLATE = [r"^\s*Skip to main content\s*$", r"^\s*Page last reviewed:.*$",
               r"^\s*Next review due:.*$", r"^\s*Credit:.*$", r"^\s*-\s*NHS\s*$", r"https?://\S+"]

def clean_doc(txt):
    """NHS pages carry navigation, image credits and review dates. ~22% of characters."""
    for p in BOILERPLATE:
        txt = re.sub(p, " ", txt, flags=re.MULTILINE)
    return re.sub(r"[ \t]{2,}", " ", re.sub(r"\s*\n\s*", "\n", txt)).strip().lstrip("\ufeff")

def load_class_doc(disease, max_chars=4000):
    # Six labels break the convention (Bronchitis, Multiple_sclerosis, ...) -> match lowercased.
    path = os.path.join(SEARCH_DIR, disease.lower() + ".txt")
    try:
        return clean_doc(open(path, encoding="utf-8").read())[:max_chars]
    except FileNotFoundError:
        return ""

ALL_LABELS = sorted(train_df.disease.unique())

def build_index(variant):
    texts, labels = fit_df.question.tolist(), fit_df.disease.tolist()
    if "L" in variant:
        texts += [d.replace("_", " ") for d in ALL_LABELS]; labels += ALL_LABELS
    if "A" in variant:
        texts += fit_df.answer.tolist(); labels += fit_df.disease.tolist()
    if "D" in variant:
        docs = [(load_class_doc(d), d) for d in ALL_LABELS]
        texts += [t for t, _ in docs if t]; labels += [d for t, d in docs if t]
    return texts, labels

variant_rows = []
for variant in ["Q", "QL", "QLA", "QLAD"]:
    t, l = build_index(variant)
    r, _ = knn_rank(t, l, val_df.question.tolist(), k=K_BEST, vec_kwargs=VEC_BEST)
    variant_rows.append({"index_variant": variant, "index_size": len(t),
                         **score_ranked(r, val_df.disease.tolist())})

index_ablation = pd.DataFrame(variant_rows)
index_ablation.to_csv(f"{ART_DIR}/arm1_index_ablation.csv", index=False)
BEST_VARIANT = index_ablation.sort_values("accuracy", ascending=False).iloc[0].index_variant
IDX_TEXTS, IDX_LABELS = build_index(BEST_VARIANT)
print("best index variant:", BEST_VARIANT)
index_ablation

In [ ]:
# ---------------- 2.4 ABSTENTION: accuracy-coverage trade-off ----------------
ranked_val, sim_val = knn_rank(IDX_TEXTS, IDX_LABELS, val_df.question.tolist(),
                               k=K_BEST, vec_kwargs=VEC_BEST)
gold_val = val_df.disease.tolist()

cov = pd.DataFrame([
    {"threshold": round(float(t), 2),
     "coverage": round(np.mean(np.array(sim_val) >= t), 3),
     "accuracy": round(float(np.mean([r[0] == g for r, s, g in
                                      zip(ranked_val, sim_val, gold_val) if s >= t])), 4)
                 if any(s >= t for s in sim_val) else np.nan}
    for t in np.arange(0, 1.01, 0.05)])

plt.figure(figsize=(7, 5))
plt.plot(cov.coverage, cov.accuracy, marker="o")
plt.xlabel("Coverage (fraction answered)"); plt.ylabel("Accuracy on answered questions")
plt.title("Arm 1: accuracy-coverage trade-off under similarity thresholding")
plt.tight_layout(); plt.savefig(f"{FIG_DIR}/fig4_coverage.png", dpi=150); plt.show()
cov.to_csv(f"{ART_DIR}/arm1_coverage.csv", index=False); cov.head()

In [ ]:
# ---------------- 2.5 SECOND TRADITIONAL VARIANT: discriminative linear model ----------------
from sklearn.svm import LinearSVC
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline

lin_rows = []
for name, clf in [("LinearSVC", LinearSVC(C=1.0, random_state=SEED)),
                  ("LogisticRegression", LogisticRegression(max_iter=1000, n_jobs=-1,
                                                            random_state=SEED))]:
    pipe = make_pipeline(TfidfVectorizer(**VEC_BEST), clf).fit(fit_df.question, fit_df.disease)
    pred = pipe.predict(val_df.question)
    lin_rows.append({"model": name,
                     "accuracy": round(float(np.mean(pred == val_df.disease)), 4),
                     "macro_f1": round(float(f1_score(val_df.disease, pred,
                                                      average="macro", zero_division=0)), 4)})
pd.DataFrame(lin_rows)

---
## Phase 3 — Arm 2: fine-tuned BERT classifier (906-way)

Follows the Week 9 lab (`AutoModelForSequenceClassification`, `AdamW`, best-model-on-validation),
with three deliberate changes, each of which needs a justifying sentence in §2.4:

1. **`max_length=48`, not 512.** The lab classified full clinical transcriptions; here the 99th
   percentile question is 18 words. This is the reason the arm fits comfortably on a free T4.
2. **Validation loss is tracked.** The brief requires training *and* validation loss curves; the
   lab loop records training loss only.
3. **906 labels.** Label encoding is fixed once and shared by every arm.

In [ ]:
import torch
from torch.utils.data import TensorDataset, DataLoader, RandomSampler, SequentialSampler
from torch.optim import AdamW
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from tqdm.auto import tqdm

set_seed()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

CLASSES = sorted(train_df.disease.unique())
c2i = {c: i for i, c in enumerate(CLASSES)}
json.dump(CLASSES, open(f"{ART_DIR}/classes.json", "w"))

HP = dict(model_name="emilyalsentzer/Bio_ClinicalBERT",
          max_length=48,          # covers the 99th percentile of question length
          batch_size=32,
          epochs=10,
          learning_rate=2e-5,
          weight_decay=0.01)
print(len(CLASSES), "classes |", device); HP

In [ ]:
def make_loader(texts, labels, tok, shuffle, bs, max_len):
    enc = tok(list(texts), padding="max_length", truncation=True,
              max_length=max_len, return_tensors="pt")
    ds = TensorDataset(enc["input_ids"], enc["attention_mask"],
                       torch.tensor([c2i[l] for l in labels]))
    return DataLoader(ds, sampler=RandomSampler(ds) if shuffle else SequentialSampler(ds),
                      batch_size=bs)

@torch.no_grad()
def evaluate_bert(model, loader):
    model.eval(); losses, preds, golds, logits_all = [], [], [], []
    for ids, mask, y in loader:
        out = model(ids.to(device), attention_mask=mask.to(device), labels=y.to(device))
        losses.append(out.loss.item())
        lg = out.logits.detach().cpu(); logits_all.append(lg)
        preds += lg.argmax(1).tolist(); golds += y.tolist()
    return float(np.mean(losses)), np.array(preds), np.array(golds), torch.cat(logits_all).numpy()

def train_bert(hp):
    set_seed()
    tok   = AutoTokenizer.from_pretrained(hp["model_name"])
    model = AutoModelForSequenceClassification.from_pretrained(
        hp["model_name"], num_labels=len(CLASSES)).to(device)
    tr_ld = make_loader(fit_df.question, fit_df.disease, tok, True,  hp["batch_size"], hp["max_length"])
    va_ld = make_loader(val_df.question, val_df.disease, tok, False, hp["batch_size"], hp["max_length"])
    opt = AdamW(model.parameters(), lr=hp["learning_rate"], weight_decay=hp["weight_decay"])

    history, best_state, best_acc = [], None, -1.0
    for ep in range(hp["epochs"]):
        model.train(); tl = []
        for ids, mask, y in tqdm(tr_ld, desc=f"epoch {ep+1}/{hp['epochs']}", leave=False):
            model.zero_grad()
            out = model(ids.to(device), attention_mask=mask.to(device), labels=y.to(device))
            out.loss.backward(); opt.step(); tl.append(out.loss.item())
        vloss, vpred, vgold, _ = evaluate_bert(model, va_ld)
        vacc = float((vpred == vgold).mean())
        history.append({"epoch": ep + 1, "train_loss": float(np.mean(tl)),
                        "val_loss": vloss, "val_accuracy": vacc})
        print(history[-1])
        if vacc > best_acc:
            best_acc = vacc
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
    model.load_state_dict(best_state)
    return model, tok, pd.DataFrame(history)

bert_model, bert_tok, hist = train_bert(HP)
hist.to_csv(f"{ART_DIR}/arm2_history.csv", index=False); hist

In [ ]:
# ---------------- REQUIRED FIGURE: training and validation loss curves ----------------
fig, ax = plt.subplots(1, 2, figsize=(12, 4.5))
ax[0].plot(hist.epoch, hist.train_loss, marker="o", label="train")
ax[0].plot(hist.epoch, hist.val_loss,  marker="s", label="validation")
ax[0].set(xlabel="Epoch", ylabel="Cross-entropy loss", title="Arm 2: loss curves"); ax[0].legend()
ax[1].plot(hist.epoch, hist.val_accuracy, marker="o", color="green")
ax[1].set(xlabel="Epoch", ylabel="Validation accuracy", title="Arm 2: validation accuracy")
plt.tight_layout(); plt.savefig(f"{FIG_DIR}/fig5_loss_curves.png", dpi=150); plt.show()
# Interpret the gap between the curves in §4.2: divergence = memorising ~10 examples per class.

In [ ]:
# ---------------- MODEL ABLATION: general vs clinical pretraining (Week 9 Task 4) ----------------
ablation = []
for mn in ["bert-base-uncased", "emilyalsentzer/Bio_ClinicalBERT"]:
    m, t, h = train_bert({**HP, "model_name": mn})
    ablation.append({"model": mn, "best_val_accuracy": float(h.val_accuracy.max()),
                     "best_epoch": int(h.loc[h.val_accuracy.idxmax(), "epoch"])})
    del m
    if torch.cuda.is_available(): torch.cuda.empty_cache()
pd.DataFrame(ablation)

---
## Phase 4 — Arm 3: LLM selection over a retrieval-shortlisted candidate set

All 906 labels in one prompt costs ~3k tokens and degrades selection. Instead **Arm 1 shortlists
20 candidates and the LLM chooses among them** — the Week 10 `diagnosis_selection_prompt` pattern
scaled to a realistic label space, and a genuine hybrid worth discussing in §4.1.

Three prompt conditions, so that §4.2's reflection on prompt design is evidenced rather than
asserted: **zero-shot**, **few-shot** (5 retrieved exemplars), **zero-shot CoT** (Kojima et al.,
2022). 200 questions × 3 conditions = 600 generations, cheap either way.

Note the shortlist ceiling: Arm 1's acc@20 bounds Arm 3 from above. Report that bound — it makes
Arm 3's numbers interpretable instead of mysterious.

In [ ]:
USE_API = False        # True -> Replicate path from the Week 10 lab (no GPU needed)

if USE_API:
    !pip -q install replicate
    import replicate as _rep
    os.environ["REPLICATE_API_TOKEN"] = "<YOUR_TOKEN>"
    _client = _rep.Client(api_token=os.environ["REPLICATE_API_TOKEN"])
    def pipe(messages, max_new_tokens=32):
        return "".join(_client.run("qwen/qwen3-235b-a22b-instruct-2507",
                    input={"prompt": "".join(m["content"] for m in messages),
                           "max_tokens": max_new_tokens, "temperature": 0}))
else:
    from transformers import AutoModelForCausalLM
    LLM_ID  = "microsoft/MediPhi-Guidelines"          # 3.8B, fp16 ~7.6GB -> fits a T4
    llm_tok = AutoTokenizer.from_pretrained(LLM_ID)
    llm = AutoModelForCausalLM.from_pretrained(
        LLM_ID, torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32).to(device)

    @torch.no_grad()
    def pipe(messages, max_new_tokens=32):
        ids = llm_tok.apply_chat_template(messages, add_generation_prompt=True,
                                          return_tensors="pt").to(device)
        out = llm.generate(ids, max_new_tokens=max_new_tokens, do_sample=False,
                           pad_token_id=llm_tok.eos_token_id)   # greedy => reproducible
        return llm_tok.decode(out[0][ids.shape[1]:], skip_special_tokens=True).strip()

In [ ]:
SYSTEM = ("You are a clinical information specialist. You map a patient question to the single "
          "most relevant condition from a supplied list. You answer with the label only.")

def build_prompt(question, candidates, mode="zero_shot", exemplars=None):
    cand = "\n".join(f"- {c}" for c in candidates)
    if mode == "zero_shot":
        user = (f"Candidate conditions:\n{cand}\n\nPatient question:\n{question}\n\n"
                "Respond with exactly one label copied from the candidate list.")
    elif mode == "few_shot":
        shots = "\n".join(f"Question: {q}\nCondition: {d}" for q, d in exemplars)
        user = (f"Candidate conditions:\n{cand}\n\nExamples:\n{shots}\n\n"
                f"Question: {question}\nCondition:")
    else:  # chain of thought
        user = (f"Candidate conditions:\n{cand}\n\nPatient question:\n{question}\n\n"
                "Let's think step by step, then give the final answer on a new line prefixed "
                "with 'ANSWER: ' using exactly one label from the list.")
    return [{"role": "system", "content": SYSTEM}, {"role": "user", "content": user}]

def parse_label(raw, candidates):
    """Constrain the output by post-hoc matching; fall back to the retriever's top choice."""
    txt = raw.split("ANSWER:")[-1].strip().lower()
    for c in candidates:
        if c.lower() in txt or c.replace("_", " ").lower() in txt:
            return c, False
    return candidates[0], True          # record fallback rate — it is a reportable number

In [ ]:
SHORTLIST_K, N_SHOTS = 20, 5

def run_arm3(eval_df, mode):
    preds, fallbacks = [], 0
    ranked, _ = knn_rank(IDX_TEXTS, IDX_LABELS, eval_df.question.tolist(),
                         k=50, vec_kwargs=VEC_BEST)
    nbrs, _   = knn_rank(fit_df.question.tolist(), list(fit_df.index),
                         eval_df.question.tolist(), k=N_SHOTS, vec_kwargs=VEC_BEST)
    for q, cands, nb in tqdm(list(zip(eval_df.question, ranked, nbrs)), desc=mode):
        cands = cands[:SHORTLIST_K]
        shots = [(fit_df.loc[i, "question"], fit_df.loc[i, "disease"]) for i in nb[:N_SHOTS]] \
                if mode == "few_shot" else None
        raw = pipe(build_prompt(q, cands, mode, shots),
                   max_new_tokens=256 if mode == "cot" else 32)
        p, fb = parse_label(raw, cands); preds.append(p); fallbacks += fb
    return preds, fallbacks

# Shortlist ceiling: Arm 3 cannot beat this.
r20, _ = knn_rank(IDX_TEXTS, IDX_LABELS, val_df.question.tolist(), k=50, vec_kwargs=VEC_BEST)
print("shortlist ceiling (gold in top-20):",
      round(float(np.mean([g in r[:SHORTLIST_K] for r, g in zip(r20, gold_val)])), 3))

arm3_val = {}
for mode in ["zero_shot", "few_shot", "cot"]:
    p, fb = run_arm3(val_df, mode)
    arm3_val[mode] = {"accuracy": round(float(np.mean(np.array(p) == val_df.disease.values)), 4),
                      "retriever_fallbacks": fb}
arm3 = pd.DataFrame(arm3_val).T
arm3.to_csv(f"{ART_DIR}/arm3_prompt_ablation.csv"); arm3

---
## Phase 5 — Test evaluation, significance testing, error analysis

**Run the test set once**, with every hyperparameter frozen from validation. With n=200 the 95%
interval on accuracy is roughly ±7pp, so a raw difference between two systems is not evidence on
its own — hence bootstrap intervals and a paired McNemar test.

In [ ]:
from scipy.stats import binomtest

def bootstrap_ci(correct, n_boot=5000, seed=SEED):
    c = np.asarray(correct, float); rng = np.random.default_rng(seed)
    boots = np.array([c[rng.integers(0, len(c), len(c))].mean() for _ in range(n_boot)])
    return (round(float(c.mean()), 4), round(float(np.percentile(boots, 2.5)), 4),
            round(float(np.percentile(boots, 97.5)), 4))

def mcnemar_exact(correct_a, correct_b):
    """Paired test on discordant pairs — the correct test for two systems on one test set."""
    a, b = np.asarray(correct_a, bool), np.asarray(correct_b, bool)
    n01, n10 = int((a & ~b).sum()), int((~a & b).sum())
    p = binomtest(n01, n01 + n10, 0.5).pvalue if (n01 + n10) else 1.0
    return {"a_only": n01, "b_only": n10, "n_discordant": n01 + n10, "p_value": round(float(p), 5)}

In [ ]:
# ---------------- FROZEN TEST RUN ----------------
gold_test = test_df.disease.tolist()
test_preds = {}

# Arm 1 — refit the index on ALL training data with the frozen configuration.
def build_index_full(variant):
    texts, labels = train_df.question.tolist(), train_df.disease.tolist()
    if "L" in variant:
        texts += [d.replace("_", " ") for d in ALL_LABELS]; labels += ALL_LABELS
    if "A" in variant:
        texts += train_df.answer.tolist(); labels += train_df.disease.tolist()
    if "D" in variant:
        docs = [(load_class_doc(d), d) for d in ALL_LABELS]
        texts += [t for t, _ in docs if t]; labels += [d for t, d in docs if t]
    return texts, labels

FULL_TEXTS, FULL_LABELS = build_index_full(BEST_VARIANT)
ranked_test, sim_test = knn_rank(FULL_TEXTS, FULL_LABELS, test_df.question.tolist(),
                                 k=K_BEST, vec_kwargs=VEC_BEST)
test_preds["Arm1_TFIDF_kNN"] = [r[0] for r in ranked_test]

# Arm 2
test_loader = make_loader(test_df.question, test_df.disease, bert_tok, False,
                          HP["batch_size"], HP["max_length"])
_, bpred, _, _ = evaluate_bert(bert_model, test_loader)
test_preds["Arm2_BERT"] = [CLASSES[i] for i in bpred]

# Arm 3 — winning prompt condition only
BEST_MODE = arm3.accuracy.idxmax()
test_preds[f"Arm3_LLM_{BEST_MODE}"], _ = run_arm3(test_df, BEST_MODE)

summary = []
for name, preds in test_preds.items():
    correct = [p == g for p, g in zip(preds, gold_test)]
    acc, lo, hi = bootstrap_ci(correct)
    summary.append({"system": name, "accuracy": acc, "ci95": f"[{lo}, {hi}]",
                    "macro_f1": round(float(f1_score(gold_test, preds,
                                                     average="macro", zero_division=0)), 4)})
summary = pd.DataFrame(summary)
summary.to_csv(f"{ART_DIR}/test_results.csv", index=False)

for a, b in itertools.combinations(test_preds, 2):
    print(a, "vs", b, mcnemar_exact([p == g for p, g in zip(test_preds[a], gold_test)],
                                    [p == g for p, g in zip(test_preds[b], gold_test)]))
summary

### 5.1 Error analysis

A 906×906 confusion matrix is unreadable, so substitute two views and justify the substitution:
the most-confused class pairs, and errors bucketed by cause. The brief requires **worked examples
of correct and incorrect predictions per method** — two of each per arm, quoted verbatim.

The key question to answer here: how many errors are *within a label family* (`baby_*` predicted
as another `baby_*`)? Those are granularity artefacts rather than clinical misunderstanding, and
that distinction drives the §4.3 argument about deployment.

In [ ]:
def confusion_pairs(preds, gold, top=15):
    return pd.Series([f"{g} -> {p}" for g, p in zip(gold, preds) if g != p]).value_counts().head(top)

def family_error_breakdown(preds, gold):
    fam_of = lambda s: s.split("_")[0]
    err = [(g, p) for g, p in zip(gold, preds) if g != p]
    same = sum(fam_of(g) == fam_of(p) for g, p in err)
    return {"n_errors": len(err), "same_family_errors": same,
            "pct_same_family": round(100 * same / len(err), 1) if err else 0.0}

def worked_examples(preds, gold, questions, n=2):
    df = pd.DataFrame({"question": questions, "true": gold, "pred": preds})
    df["correct"] = df.true == df.pred
    return df[df.correct].head(n), df[~df.correct].head(n)

for name, preds in test_preds.items():
    print(f"\n===== {name} =====")
    print(family_error_breakdown(preds, gold_test))
    ok, bad = worked_examples(preds, gold_test, test_df.question.tolist())
    print("CORRECT:\n", ok.to_string(index=False))
    print("INCORRECT:\n", bad.to_string(index=False))

In [ ]:
# ---------------- Top-15 confusion heat map ----------------
top_pairs = confusion_pairs(test_preds["Arm1_TFIDF_kNN"], gold_test, top=15)
plt.figure(figsize=(7, 6))
sns.barplot(y=top_pairs.index, x=top_pairs.values, orient="h")
plt.xlabel("Count"); plt.title("Most frequent confusions (Arm 1, test set)")
plt.tight_layout(); plt.savefig(f"{FIG_DIR}/fig6_confusions.png", dpi=150); plt.show()

---
## Reproducibility checklist

- [ ] `SEED = 42`; `set_seed()` re-called before every training run
- [ ] Splits written to `artefacts/` and reloaded, never regenerated inline
- [ ] Versions and GPU recorded in `artefacts/environment.json`
- [ ] LLM decoding greedy (`do_sample=False`)
- [ ] Test set evaluated once, after hyperparameters were frozen on validation
- [ ] Wall-clock time and peak GPU memory recorded per arm for §4.1
- [ ] Notebook restarted and run top-to-bottom before submission